# NumPy Arrays

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 1/7

Python lists are flexible but slow for number crunching. NumPy gives you arrays that store numbers the way the CPU likes them — understanding that one idea explains most of scientific Python.

## 🎯 Learning Objectives

- Install NumPy and adopt the universal `import numpy as np` convention
- Explain why NumPy arrays beat Python lists for numerical work
- Measure the speed gap between a Python loop and a vectorized operation
- Create arrays from nested lists with `np.array`
- Inspect any array using `ndim`, `shape`, `size`, `dtype` and `itemsize`
- Build arrays from scratch with `zeros`, `ones`, `full`, `empty`, `arange`, `linspace` and `eye`, and choose correctly between `arange` and `linspace`

## 1. Installing NumPy

NumPy is not part of the standard library, so install it once into your virtual environment. Almost the entire scientific Python stack — pandas, scikit-learn, PyTorch — is built on top of it.

**Syntax:**
```bash
pip install numpy
```

After installing, import it everywhere with the alias `np`. This is a worldwide convention — every tutorial, library and colleague assumes it.

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)
print("The convention: import numpy as np - never anything else.")

## 2. Why Does NumPy Exist?

A Python list is really a list of *references* to Python objects scattered around memory. To double every element, Python must chase each pointer, inspect each object, check "is this even a number?", and allocate a brand-new object for the result.

A NumPy array instead packs raw numbers back-to-back in one contiguous block of memory and applies operations with compiled C loops — no pointer chasing, no type checks, no per-element allocations.

**Rule of thumb:** lists are for *collections of things*; arrays are for *numbers you compute with*.

**Syntax:**
```python
arr = np.array([10, 20, 30])
arr * 2        # array([20, 40, 60]) - every element at once, no loop
```

In [ ]:
import numpy as np

temps_dhaka = np.array([31.5, 32.0, 33.8, 30.2, 29.9])

# Every operation below touches EVERY element - no loops anywhere
print("temps        :", temps_dhaka)
print("+2 degrees   :", temps_dhaka + 2)
print("in Fahrenheit:", temps_dhaka * 9 / 5 + 32)
print("above 32?    :", temps_dhaka > 32)

## 3. Speed Test: Python Loop vs NumPy

Talking is cheap — let's measure. We will sum the same one million numbers three ways: a manual `for` loop, Python's built-in `sum()`, and NumPy's `np.sum()`.

**Syntax:**
```python
import time

start = time.perf_counter()
# ... some work ...
elapsed = time.perf_counter() - start
```

In [ ]:
import time
import numpy as np

N = 1_000_000                      # one million elements - a fixed, fair race
py_list = list(range(N))           # classic Python list
np_array = np.arange(N)            # the same numbers as a NumPy array

print("list :", type(py_list).__name__, "| length:", len(py_list))
print("array:", type(np_array).__name__, "| length:", np_array.size)
print("Same total?", sum(py_list) == np.sum(np_array))

In [ ]:
import time
import numpy as np

N = 1_000_000
py_list = list(range(N))
np_array = np.arange(N)

start = time.perf_counter()
total = 0
for x in py_list:                  # pure-Python loop
    total += x
loop_time = time.perf_counter() - start

start = time.perf_counter()
total = sum(py_list)               # built-in: C-speed iteration, but still full Python objects
builtin_time = time.perf_counter() - start

start = time.perf_counter()
total = np.sum(np_array)           # compiled C loop over contiguous machine ints
numpy_time = time.perf_counter() - start

print(f"for-loop : {loop_time * 1000:9.2f} ms")
print(f"sum()    : {builtin_time * 1000:9.2f} ms")
print(f"np.sum() : {numpy_time * 1000:9.2f} ms")
print(f"NumPy was roughly {loop_time / max(numpy_time, 1e-9):.0f}x faster than the loop")

> 🔍 **Under the Hood:** Why is NumPy often 50–100× faster?
> - A Python `int` is a full object: reference counter, type pointer, digit payload — about 28 bytes for a small number, PLUS an 8-byte pointer to it inside the list.
> - A NumPy `int64` array is exactly 8 bytes per number, laid out back-to-back in one contiguous buffer.
> - Contiguity lets the CPU stream through the data sequentially with warm cache lines, and NumPy's inner loops are compiled C using SIMD instructions that process several numbers per CPU cycle.
> - The interpreter, meanwhile, executes bytecode dispatches to `int.__add__`, checks types, and allocates a fresh result object *for every single element*.
>
> Same math, radically different machinery. "Vectorize your code" is the #1 performance rule in data science for exactly this reason.

## 4. Vectorization: Think in Whole Arrays

**Vectorization** means expressing an operation on entire arrays instead of element-by-element loops. You describe *what* you want ("double everything"); NumPy decides *how* to iterate (in C). Less code, fewer bugs, dramatically faster.

**Syntax:**
```python
arr = arr * 2            # scale every element
arr = arr + offset       # shift every element
result = arr_a + arr_b   # element-wise between two same-shaped arrays
```

In [ ]:
import numpy as np

prices_list = [120, 250, 80, 430, 95]            # Python list
prices = np.array(prices_list)                   # NumPy array

with_vat_list = [p * 1.15 for p in prices_list]  # old habit: loop over elements
with_vat_np = prices * 1.15                      # NumPy: one expression

print("comprehension:", with_vat_list)
print("vectorized   :", with_vat_np)

# Array-to-array math is element-wise too
fees = np.array([5, 10, 5, 10, 5])
print("prices + fees :", prices + fees)

## 5. Creating Arrays from Lists: np.array

`np.array()` accepts any list of numbers — and nested lists become dimensions. A list of lists becomes a 2-D array (rows × columns), exactly like a spreadsheet. Three levels deep gives a 3-D array (think stacks of spreadsheets, or image batches).

**Syntax:**
```python
v = np.array([1, 2, 3])            # 1-D: a vector
m = np.array([[1, 2], [3, 4]])     # 2-D: rows of equal length
t = np.array([[[1, 2], [3, 4]]])   # 3-D and beyond
```

In [ ]:
import numpy as np

vector = np.array([10, 20, 30])
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
cube = np.array([[[1, 2], [3, 4]],
                 [[5, 6], [7, 8]]])

print("vector:", vector)
print("matrix:")
print(matrix)
print("cube shape:", cube.shape)

## 6. Core Attributes: ndim, shape, size, dtype, itemsize

Every array carries metadata describing exactly what it is. Memorize these five — you will read them constantly while debugging any data or ML code.

**Syntax:**
```python
arr.ndim       # number of axes (dimensions)
arr.shape      # length along each axis, e.g. (2, 3)
arr.size       # total element count (product of shape)
arr.dtype      # data type stored in each slot
arr.itemsize   # bytes occupied by one element
```

In [ ]:
import numpy as np

scores = np.array([[78, 85, 90],
                   [62, 71, 88]])

print("scores:")
print(scores)
print()
print("ndim    :", scores.ndim)      # 2 -> two axes: rows and columns
print("shape   :", scores.shape)    # (2, 3) -> 2 rows, 3 columns
print("size    :", scores.size)      # 6 elements in total
print("dtype   :", scores.dtype)     # int64 -> each slot holds a 64-bit int
print("itemsize:", scores.itemsize)  # 8 bytes per element
print("nbytes  :", scores.nbytes)    # whole array: 6 x 8 = 48 bytes

In [ ]:
import numpy as np

# A batch of grayscale images: (batch, height, width) - the shape ML models eat
batch = np.zeros((4, 28, 28))

print("batch.ndim  :", batch.ndim)
print("batch.shape :", batch.shape)   # 4 images, each 28x28 pixels
print("batch.size  :", batch.size)    # 4 * 28 * 28 = 3136 values
print("batch.dtype :", batch.dtype)   # float64 is the default

## 7. Creation Helpers: zeros, ones, full, empty, eye

Real projects rarely type arrays by hand — they generate them. Need a blank canvas for weights? `zeros`. A flag mask? `ones` or `full`. The identity matrix for linear algebra? `eye`.

**Syntax:**
```python
np.zeros(shape)      # all 0.0
np.ones(shape)       # all 1.0
np.full(shape, val)  # all filled with val
np.empty(shape)      # UNINITIALIZED - whatever bytes are already there!
np.eye(n)            # n x n identity matrix (1s on the diagonal)
```

The shape is a single int for 1-D, or a tuple like `(3, 4)` for 2-D.

In [ ]:
import numpy as np

print("zeros((2, 3)):")
print(np.zeros((2, 3)))
print()
print("ones((3, 2)):")
print(np.ones((3, 2)))
print()
print("full((2, 2), 7):")
print(np.full((2, 2), 7))
print()
print("eye(3) - identity matrix:")
print(np.eye(3))

In [ ]:
import numpy as np

# empty does NOT zero memory - it returns whatever bytes were lying there!
scratch = np.empty((2, 2))
print("empty (unpredictable contents):")
print(scratch)

# Every helper accepts dtype= to control precision and memory usage
light = np.zeros((2, 2), dtype=np.float32)
flags = np.ones(3, dtype=bool)
print("float32 zeros dtype:", light.dtype)
print("bool ones          :", flags)

## 8. arange: Ranges as Arrays

`np.arange` is the array cousin of Python's `range`: a start, a stop (exclusive!), and a step.

**Syntax:**
```python
np.arange(stop)                # 0 .. stop-1
np.arange(start, stop)         # start .. stop-1
np.arange(start, stop, step)   # jumping by step
```

In [ ]:
import numpy as np

print(np.arange(5))          # [0 1 2 3 4]
print(np.arange(2, 8))       # [2 3 4 5 6 7]
print(np.arange(0, 21, 5))   # every 5th taka up to 20
print(np.arange(10, 0, -2))  # countdown - negative steps work too

## 9. linspace: Exactly N Evenly Spaced Points

`np.linspace(start, stop, num)` asks for a **count** of points spread evenly from start to stop — and stop is **included** by default. It is the standard way to build plot axes and sampling grids.

**Syntax:**
```python
np.linspace(start, stop, num=50)               # num points, stop included
np.linspace(start, stop, num, endpoint=False)  # exclude the stop point
```

In [ ]:
import numpy as np

print(np.linspace(0, 10, 5))                  # 0, 2.5, 5, 7.5, 10 (stop INCLUDED)
print(np.linspace(0, 1, 5))                   # quarters
print(np.linspace(0, 1, 5, endpoint=False))   # 0 .. 0.75, stop excluded

## 10. arange vs linspace — Which One?

Both produce sequences, but they answer different questions: `arange` thinks in **step size**, `linspace` thinks in **number of points**.

| Question you are asking | Use | Example |
|---|---|---|
| "Every 5th taka from 0 to 100" | `arange` | `np.arange(0, 101, 5)` |
| "Exactly 21 points from 0 to 100" | `linspace` | `np.linspace(0, 100, 21)` |

Prefer `linspace` whenever the step would be fractional — floating-point steps make `arange`'s element count unpredictable.

In [ ]:
import numpy as np

risky = np.arange(0, 1, 0.1)     # step 0.1, stop 1.0 excluded -> ends at 0.9
safe = np.linspace(0, 1, 11)     # 11 points, 1.0 GUARANTEED included

print("arange(0, 1, 0.1) :", risky, "->", risky.size, "values")
print("linspace(0, 1, 11):", safe)
print("Want 1.0 included with arange? Rounding decides - not you. Use linspace.")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Writing `for` loops over big arrays | Throws away NumPy's C speed — often 100× slower | Vectorize: `arr * 2`, `np.sum(arr)` |
| Assuming `np.empty` returns zeros | It returns leftover garbage from memory | Use `np.zeros` unless you will overwrite every element |
| Nesting lists with unequal lengths | `np.array([[1, 2], [3]])` fails or makes a broken object array | Keep rows the same length; pad ragged data first |
| Using `arange` with float steps | The endpoint and count depend on rounding errors | Prefer `linspace` for fractional spacing |
| Mixing list and array arithmetic casually | `[1, 2] + [3, 4]` concatenates, arrays add element-wise | Convert both sides with `np.array(...)` before computing |

❌ `[1, 2] + [3, 4]` → `[1, 2, 3, 4]` (concatenation!)
✅ `np.array([1, 2]) + np.array([3, 4])` → `array([4, 6])` (element-wise math)

## 💡 Best Practices & Pro Tips

- Always `import numpy as np` — every codebase, tutorial and interviewer assumes this alias.
- Print `.shape` and `.dtype` right after creating or transforming an array; most data-code bugs are shape bugs.
- Use `np.arange` for integer sequences, `np.linspace` for plot axes and sampling grids.
- Pass explicit dtypes (`dtype=np.float32`) when arrays feed ML frameworks — many require float32 to save memory and GPU bandwidth.
- **AI-engineering relevance:** tensors in PyTorch and TensorFlow ARE NumPy-style arrays. `.shape`, `.dtype`, creation helpers and vectorization transfer 1:1 — mastering this lesson is half of understanding model internals.

## 📌 Summary

| Tool / Attribute | What it does | Example |
|---|---|---|
| `np.array(list)` | Build an array from (nested) lists | `np.array([[1, 2], [3, 4]])` |
| `.ndim` | Number of axes | `m.ndim` → `2` |
| `.shape` | Length along each axis | `m.shape` → `(2, 3)` |
| `.size` | Total element count | `m.size` → `6` |
| `.dtype` / `.itemsize` | Element type / bytes per element | `m.dtype` → `int64` |
| `np.zeros(shape)` | All zeros | `np.zeros((2, 3))` |
| `np.ones(shape)` | All ones | `np.ones(4)` |
| `np.full(shape, v)` | Filled with value v | `np.full((2, 2), 7)` |
| `np.empty(shape)` | Uninitialized scratch memory | `np.empty(3)` |
| `np.eye(n)` | Identity matrix | `np.eye(3)` |
| `np.arange(s, e, step)` | Range array, stop exclusive | `np.arange(0, 10, 2)` |
| `np.linspace(s, e, n)` | n evenly spaced points, ends included | `np.linspace(0, 1, 5)` |

Key takeaways:
- NumPy arrays store homogeneous numbers in contiguous memory and compute with compiled C loops — that is the entire magic trick.
- Vectorize: replace Python loops with whole-array expressions.
- `shape` and `dtype` are the first two things to inspect when anything behaves oddly.
- `arange` is step-driven; `linspace` is count-driven.

## 🔗 Next Lesson

- Continue to **[02_Indexing_Slicing](../02_Indexing_Slicing/notes.ipynb)** — selecting exactly the elements you need, plus the view-vs-copy trap that bites every beginner once.